In [ ]:
import os
from pathlib import Path
import joblib
import pandas as pd
pd.set_option('display.max_columns', None)
import plotly.express as px
from plotly.subplots import make_subplots

# Parametros

In [ ]:
ambiente = 'project'
costa = 'Matamoros'
PORCENTAJE_SAMPLE_DATA = 0.7
EVERY_N_YEARS = 2
RANDOM_SEED = 0
SCALER_FEATURES = [
    'wind_speed_ms', 'wind_cos_direction', 'wind_sin_direction', 'wave_height_m', 
    'wave_cos_direction', 'wave_sin_direction', 'wave_period_s', 'wave_energy', 'wave_steepness'
]
DBSCAN_FEATURES = [
    'wind_speed_ms', 'wave_energy', 
    'wave_period_s', 'wave_steepness'
]
GMM_FEATURES = [
    'wind_speed_ms', 'wave_energy', 
    'wave_period_s', 'wave_steepness'
]
RF_FEATURES = [
    'wind_speed_ms', 'wave_energy', 
    'wave_period_s', 'wave_steepness'
]
EXTREME_FEATURES = ['wind_speed_ms', 'wave_energy', 'wave_period_s']
SORT_FEATURES = ["wave_energy", "wind_speed_ms", "wave_steepness"]
EXTREME_THRESHOLD = 0.99
if 'DATABRICKS_RUNTIME_VERSION' in os.environ:
    scaler_path = f'/Volumes/cor_{ambiente}/ml/models/scaler/scaler_{{}}.pkl'
    model_path = f'/Volumes/cor_{ambiente}/ml/models/wave_clasificator/wave_clasificator_{{}}.pkl'
else:
    base_path = Path.cwd().parent
    scaler_path = f'{base_path}/scaler/scaler_{{}}.pkl'
    model_path = f'{base_path}/wave_clasificator/wave_clasificator_{{}}.pkl'
    

# Obtener datos

In [ ]:
data = (
        spark.sql(
            f"""
                SELECT coast_name, datetime, year,
                {', '.join(SCALER_FEATURES)},
                CONCAT(coast_name, '_', DATE_FORMAT(datetime, 'yyyyMM')) AS coast_year_month
                FROM cor_{ambiente}.silver.swell_metrics
                WHERE coast_name = '{costa}'
                AND YEAR(datetime) % {EVERY_N_YEARS} = 0
            """
        )
    )

In [ ]:
data_df = data.toPandas()

In [ ]:
data_df['year'].value_counts().sort_index()

In [ ]:
coast_year_month_dict = {row.coast_year_month: PORCENTAJE_SAMPLE_DATA for row in data.select('coast_year_month').distinct().collect()}

data_sample = (
    data
    .sampleBy('coast_year_month', fractions=coast_year_month_dict, seed=RANDOM_SEED)
    .drop('coast_year_month')
).toPandas()

In [ ]:
fig = px.scatter_matrix(
    data_sample, 
    dimensions=SCALER_FEATURES
)
fig.update_layout(
    title=f'Correlación de variables de la costa {costa}',
    width=1200, 
    height=1500
)
fig.show()

In [ ]:
correlation_matrix = data_sample[SCALER_FEATURES].corr()
fig = px.imshow(correlation_matrix, text_auto=True, aspect="auto")
fig.update_layout(
    title=f'Matriz de correlación de variables de la costa {costa}'
)
fig.show()

# Escalar

In [ ]:
scaler_path = scaler_path.format(costa)
scaler = joblib.load(scaler_path)

In [ ]:
scaled_data = scaler.transform(data_sample[SCALER_FEATURES])
scaled_df = pd.DataFrame(scaled_data, columns=SCALER_FEATURES)

# DBSCAN

In [ ]:
from sklearn.cluster import DBSCAN

In [ ]:
EPSILON = 0.05
MIN_SAMPLES = 10

In [ ]:
dbscan = DBSCAN(
    eps=EPSILON, 
    min_samples=MIN_SAMPLES
)

In [ ]:
dbscan.fit(scaled_df[DBSCAN_FEATURES])
mask_outliers = dbscan.labels_ == -1
for feature in EXTREME_FEATURES:
    mask_outliers &= data_sample[feature] > data_sample[feature].quantile(EXTREME_THRESHOLD)

data_sample['dbscan_is_outlier'] = mask_outliers

In [ ]:
data_sample['dbscan_is_outlier'].value_counts(normalize=True)*100

# Grafica

In [ ]:
fig = make_subplots(
    rows=len(EXTREME_FEATURES),
    cols=2,
    specs=[
        [{"type": "scene", "rowspan": len(EXTREME_FEATURES)}, {"type": "xy"}],
        *[
            [None, {"type": "xy"}]
            for _ in range(len(EXTREME_FEATURES) - 1)
        ]
    ],
    subplot_titles=[
        f'Muestra datos de la costa {costa}',
        *[f'Outliers por {feature}' for feature in EXTREME_FEATURES]
    ],
    vertical_spacing=0.12,
    horizontal_spacing=0.08,
    column_widths=[1/3, 2/3]
)

# Gráfica 3D
fig_3d = px.scatter_3d(
    data_sample,
    x='wind_speed_ms',
    y='wave_period_s',
    z='wave_energy',
    color='dbscan_is_outlier',
    color_discrete_map={
        False: 'blue',
        True: 'red'
    }
)

fig_3d.update_traces(marker=dict(size=3))

for trace in fig_3d.data:
    fig.add_trace(trace, row=1, col=1)

# Gráficas 2D
for i, feature in enumerate(EXTREME_FEATURES):
    tmp_fig = px.scatter(
        data_sample,
        x='datetime',
        y=feature,
        color='dbscan_is_outlier',
        color_discrete_map={
            False: 'blue',
            True: 'red'
        }
    )

    tmp_fig.update_traces(marker=dict(size=3))

    for trace in tmp_fig.data:
        trace.showlegend = False
        fig.add_trace(trace, row=i+1, col=2)

    threshold = data_sample[feature].quantile(EXTREME_THRESHOLD)

    # Línea horizontal
    fig.add_shape(
        type="line",
        x0=data_sample['datetime'].min(),
        x1=data_sample['datetime'].max(),
        y0=threshold,
        y1=threshold,
        line=dict(
            color="black",
            width=2,
            dash="dash"
        ),
        row=i+1,
        col=2
    )

fig.update_layout(
    height=len(EXTREME_FEATURES)*250,
    width=1300,
    title=f'Análisis de outliers - Costa {costa}',
    uirevision='constant',
    showlegend=False,
    margin=dict(t=90),
    scene=dict(
        xaxis_title='Velocidad del viento (m/s)',
        yaxis_title='Período de la ola (s)',
        zaxis_title='Energía de la ola (J)',
        aspectmode='cube'
    )
)
pct_outlier = (data_sample['dbscan_is_outlier'].value_counts(normalize=True)*100).loc[True]
fig.add_annotation(
    text=f"pct es outlier: {pct_outlier:.2f}%",
    xref="paper",
    yref="paper",
    x=0,
    y=-0.1,
    showarrow=False,
    font=dict(size=14)
)

fig.show()

# Gaussian Mixture

In [ ]:
from sklearn.mixture import GaussianMixture

In [ ]:
N_CLUSTERS = 6

In [ ]:
gmm = GaussianMixture(
    n_components=N_CLUSTERS,
    covariance_type="full",
    random_state=RANDOM_SEED
)

In [ ]:
data_sample["gmm_cluster"] = gmm.fit_predict(scaled_df[GMM_FEATURES])
data_sample["gmm_cluster_probability"] = gmm.predict_proba(scaled_df[GMM_FEATURES]).max(axis=1)

In [ ]:
data_sample["gmm_cluster_probability"].groupby(data_sample["gmm_cluster"]).describe()

In [ ]:
cluster_summary = (
    data_sample.groupby("gmm_cluster")[GMM_FEATURES]
    .mean()
    .sort_values(SORT_FEATURES)
)

cluster_order = {
    old_cluster: new_cluster + 1
    for new_cluster, old_cluster in enumerate(cluster_summary.index)
}

data_sample["gmm_sea_state_level"] = data_sample["gmm_cluster"].map(cluster_order)

data_sample['gmm_mask_extremo'] = data_sample[EXTREME_FEATURES[0]] > data_sample[EXTREME_FEATURES[0]].quantile(EXTREME_THRESHOLD)
for feature in EXTREME_FEATURES[1:]:
    data_sample['gmm_mask_extremo'] &= data_sample[feature] > data_sample[feature].quantile(EXTREME_THRESHOLD)

data_sample.loc[data_sample['gmm_mask_extremo'], 'gmm_sea_state_level'] = 7

In [ ]:
data_sample['gmm_sea_state_level'].value_counts(normalize=True)*100

In [ ]:
sea_state_names = {
    1: "Mar calmado",
    2: "Mar suave",
    3: "Mar dinámico",
    4: "Mar agitado",
    5: "Mar fuerte",
    6: "Mar peligroso",
    7: "Mar extremo"
}

data_sample["gmm_sea_state"] = data_sample["gmm_sea_state_level"].map(sea_state_names)

In [ ]:
data_sample['gmm_sea_state'].value_counts(normalize=True)*100

# Grafica

In [ ]:
fig = make_subplots(
    rows=len(EXTREME_FEATURES),
    cols=2,
    specs=[
        [{"type": "scene", "rowspan": len(EXTREME_FEATURES)}, {"type": "xy"}],
        *[
            [None, {"type": "xy"}]
            for _ in range(len(EXTREME_FEATURES) - 1)
        ]
    ],
    subplot_titles=[
        f'Muestra datos de la costa {costa}',
        *[f'Outliers por {feature}' for feature in EXTREME_FEATURES]
    ],
    vertical_spacing=0.12,
    horizontal_spacing=0.08,
    column_widths=[1/3, 2/3]
)

# Gráfica 3D
fig_3d = px.scatter_3d(
    data_sample,
    x='wind_speed_ms',
    y='wave_period_s',
    z='wave_energy',
    color='gmm_sea_state',
    color_discrete_map={
        False: 'blue',
        True: 'red'
    }
)

fig_3d.update_traces(marker=dict(size=3))

for trace in fig_3d.data:
    fig.add_trace(trace, row=1, col=1)

# Gráficas 2D
for i, feature in enumerate(EXTREME_FEATURES):
    tmp_fig = px.scatter(
        data_sample,
        x='datetime',
        y=feature,
        color='gmm_sea_state',
        color_discrete_map={
            False: 'blue',
            True: 'red'
        }
    )

    tmp_fig.update_traces(marker=dict(size=3))

    for trace in tmp_fig.data:
        trace.showlegend = False
        fig.add_trace(trace, row=i+1, col=2)

    threshold = data_sample[feature].quantile(EXTREME_THRESHOLD)

    # Línea horizontal
    fig.add_shape(
        type="line",
        x0=data_sample['datetime'].min(),
        x1=data_sample['datetime'].max(),
        y0=threshold,
        y1=threshold,
        line=dict(
            color="black",
            width=2,
            dash="dash"
        ),
        row=i+1,
        col=2
    )

fig.update_layout(
    height=len(EXTREME_FEATURES)*250,
    width=1300,
    title=f'Análisis de outliers - Costa {costa}',
    uirevision='constant',
    showlegend=False,
    margin=dict(t=90),
    scene=dict(
        xaxis_title='Velocidad del viento (m/s)',
        yaxis_title='Período de la ola (s)',
        zaxis_title='Energía de la ola (J)',
        aspectmode='cube'
    )
)

pct_outlier = (data_sample['gmm_sea_state'].value_counts(normalize=True)*100).loc['Mar extremo']
fig.add_annotation(
    text=f"pct es outlier: {pct_outlier:.2f}%",
    xref="paper",
    yref="paper",
    x=0,
    y=-0.1,
    showarrow=False,
    font=dict(size=14)
)

fig.show()

: 

# Random Forest

In [ ]:
import joblib
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score
)

In [ ]:
TARGET = "gmm_sea_state_level"

In [ ]:
X = data_sample[RF_FEATURES]
y = data_sample[TARGET].astype(int)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

In [ ]:
rf_classifier = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_classifier.fit(X_train, y_train)

In [ ]:
y_pred = rf_classifier.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Balanced accuracy:", balanced_accuracy_score(y_test, y_pred))

print(
    classification_report(
        y_test,
        y_pred,
        digits=3
    )
)

cm = confusion_matrix(y_test, y_pred)
print(cm)

In [ ]:
feature_importance = pd.DataFrame({
    "feature": RF_FEATURES,
    "importance": rf_classifier.feature_importances_
}).sort_values("importance", ascending=False)

display(feature_importance)

In [ ]:
model_path = model_path.format(costa)
joblib.dump(rf_classifier, model_path)

In [ ]:
data_sample["rf_sea_state_level"] = rf_classifier.predict(X)
data_sample["rf_sea_state"] = data_sample["rf_sea_state_level"].map(sea_state_names)

# Grafica

In [ ]:
fig = make_subplots(
    rows=len(EXTREME_FEATURES),
    cols=2,
    specs=[
        [{"type": "scene", "rowspan": len(EXTREME_FEATURES)}, {"type": "xy"}],
        *[
            [None, {"type": "xy"}]
            for _ in range(len(EXTREME_FEATURES) - 1)
        ]
    ],
    subplot_titles=[
        f'Muestra datos de la costa {costa}',
        *[f'Outliers por {feature}' for feature in EXTREME_FEATURES]
    ],
    vertical_spacing=0.12,
    horizontal_spacing=0.08,
    column_widths=[1/3, 2/3]
)

# Gráfica 3D
fig_3d = px.scatter_3d(
    data_sample,
    x='wind_speed_ms',
    y='wave_period_s',
    z='wave_energy',
    color='rf_sea_state',
    color_discrete_map={
        False: 'blue',
        True: 'red'
    }
)

fig_3d.update_traces(marker=dict(size=3))

for trace in fig_3d.data:
    fig.add_trace(trace, row=1, col=1)

# Gráficas 2D
for i, feature in enumerate(EXTREME_FEATURES):
    tmp_fig = px.scatter(
        data_sample,
        x='datetime',
        y=feature,
        color='rf_sea_state',
        color_discrete_map={
            False: 'blue',
            True: 'red'
        }
    )

    tmp_fig.update_traces(marker=dict(size=3))

    for trace in tmp_fig.data:
        trace.showlegend = False
        fig.add_trace(trace, row=i+1, col=2)

    threshold = data_sample[feature].quantile(EXTREME_THRESHOLD)

    # Línea horizontal
    fig.add_shape(
        type="line",
        x0=data_sample['datetime'].min(),
        x1=data_sample['datetime'].max(),
        y0=threshold,
        y1=threshold,
        line=dict(
            color="black",
            width=2,
            dash="dash"
        ),
        row=i+1,
        col=2
    )

fig.update_layout(
    height=len(EXTREME_FEATURES)*250,
    width=1300,
    title=f'Análisis de outliers - Costa {costa}',
    uirevision='constant',
    showlegend=False,
    margin=dict(t=90),
    scene=dict(
        xaxis_title='Velocidad del viento (m/s)',
        yaxis_title='Período de la ola (s)',
        zaxis_title='Energía de la ola (J)',
        aspectmode='cube'
    )
)

pct_outlier = (data_sample['rf_sea_state'].value_counts(normalize=True)*100).loc['Mar extremo']
fig.add_annotation(
    text=f"pct es outlier: {pct_outlier:.2f}%",
    xref="paper",
    yref="paper",
    x=0,
    y=-0.1,
    showarrow=False,
    font=dict(size=14)
)

fig.show()